In [1]:
import os
os.environ['GRADIENT_ACCESS_TOKEN'] = "O6P5axc9LKyV7Rzk07MCUvACI5SjHg42"
os.environ['GRADIENT_WORKSPACE_ID'] = "ebc85182-b8e8-4009-a640-fd946d6b89ed_workspace"

In [11]:
import bitsandbytes as bnb
import torch
import torch.nn as nn
import torch.optim as optim

# Print Bitsandbytes version
# print("Bitsandbytes version:", bnb.__version__)

# Check CUDA availability in PyTorch
if torch.cuda.is_available():
    print("CUDA is available.")
    print("CUDA version:", torch.version.cuda)
    print("Number of CUDA devices:", torch.cuda.device_count())
    print("CUDA device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    print("CUDA is not available.")
    exit()

# Define a simple neural network
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc = nn.Linear(10, 1)
    
    def forward(self, x):
        return self.fc(x)

# Initialize the network and move it to GPU
model = SimpleNet().cuda()
criterion = nn.MSELoss()

# Use bitsandbytes 8-bit optimizer
optimizer = bnb.optim.Adam8bit(model.parameters(), lr=0.001)

# Create dummy data
inputs = torch.randn(5, 10).cuda()
targets = torch.randn(5, 1).cuda()

# Forward pass
outputs = model(inputs)
loss = criterion(outputs, targets)

# Backward pass and optimization
optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Bitsandbytes 8-bit optimizer executed successfully.")


CUDA is available.
CUDA version: 11.8
Number of CUDA devices: 1
CUDA device name: NVIDIA GeForce RTX 3050 Laptop GPU


AttributeError: module 'bitsandbytes' has no attribute 'optim'

In [10]:
from gradientai import Gradient

def main():
  with Gradient() as gradient:
      base_model = gradient.get_base_model(base_model_slug="nous-hermes2")

      new_model = base_model.create_model_adapter(
          name="test model 3"
      )
      print(f"Created model adapter with id {new_model.id}")
      sample_query = "### Instruction: Who is the batter in Over 0 ball 1? \n\n### Response:"
      print(f"Asking: {sample_query}")

      # before fine-tuning
      completion = new_model.complete(query=sample_query, max_generated_token_count=100).generated_output
      print(f"Generated (before fine-tune): {completion}")

      samples = [
        { "inputs": "### Instruction: Over 0 ball 1 \n\n### Response: batter: RG Sharma" },
        { "inputs": "### Instruction: Over 0 ball 1 \n\n### Response: bowler: MA Starc" },
        { "inputs": "### Instruction: Over 0 ball 1 \n\n### Response: non_striker: V Kohli" },
      ]

      # this is where fine-tuning happens
      # num_epochs is the number of times you fine-tune the model
      # more epochs tends to get better results, but you also run the risk of "overfitting"
      # play around with this number to find what works best for you
      num_epochs = 3
      count = 0
      while count < num_epochs:
          print(f"Fine-tuning the model, iteration {count + 1}")
          new_model.fine_tune(samples=samples)
          count = count + 1

      # after fine-tuning
      completion = new_model.complete(query=sample_query, max_generated_token_count=100).generated_output
      print(f"Generated (after fine-tune): {completion}")

      new_model.delete()

if __name__ == "__main__":
    main()

Created model adapter with id fb7b6c80-f098-4532-9d09-15b3ef5d7596_model_adapter
Asking: ### Instruction: Who is the batter in Over 0 ball 1? 

### Response:
Generated (before fine-tune):  The batter in Over 0 ball 1 is not mentioned in the given input. Can you please provide more information or context about the situation or match?
Fine-tuning the model, iteration 1
Fine-tuning the model, iteration 2
Fine-tuning the model, iteration 3
Generated (after fine-tune):  The batter in Over 0 ball 1 is not mentioned in the given input.


In [7]:
import json
from gradientai import Gradient

def main():
    with Gradient() as gradient:
        base_model = gradient.get_base_model(base_model_slug="llama2-7b-chat")

        new_model = base_model.create_model_adapter(
            name="test model 3"
        )
        print(f"Created model adapter with id {new_model.id}")

        sample_query = "### Instruction: What happened in Over 0 Ball 1? \n\n### Response:"
        print(f"Asking: {sample_query}")

        # Before fine-tuning
        completion = new_model.complete(query=sample_query, max_generated_token_count=100).generated_output
        print(f"Generated (before fine-tune): {completion}")

        # Load samples from JSON file
        with open(r"C:\Users\Yashwanth\projects\LLM sports\formatted_samples.json", "r") as f:
            samples = json.load(f)

        # This is where fine-tuning happens
        # num_epochs is the number of times you fine-tune the model
        # More epochs tend to get better results, but you also run the risk of "overfitting"
        # Play around with this number to find what works best for you
        num_epochs = 3
        count = 0
        while count < num_epochs:
            print(f"Fine-tuning the model, iteration {count + 1}")
            # Split  samples into batches of 100
            for i in range(0, len(samples), 40):
                batch = samples[i: i + 40]
                print("Batch contents:", batch)  # Print the batch contents
                new_model.fine_tune(samples=batch) # Fine-tune with each batch
            count = count + 1

        # After fine-tuning
        completion = new_model.complete(query=sample_query, max_generated_token_count=100).generated_output
        print(f"Generated (after fine-tune): {completion}")

        new_model.delete()

if __name__ == "__main__":
    main()

Created model adapter with id cec37ebe-6a91-49ef-9753-3b7085038840_model_adapter
Asking: ### Instruction: What happened in Over 0 Ball 1? 

### Response:
Generated (before fine-tune):  I'm not sure what you're referring to. Could you please provide more context or clarify which "Over 0 Ball 1" you are asking about? I want to make sure I give you the most accurate and helpful response possible.
Fine-tuning the model, iteration 1
Batch contents: [{'inputs': '### Instruction: What happened in Over 0 Ball 1? \n\n### Response: In Over 0 Ball 1, RG Sharma faced MA Starc and scored 0 runs. Non-striker was V Kohli.'}, {'inputs': '### Instruction: What happened in Over 0 Ball 2? \n\n### Response: In Over 0 Ball 2, RG Sharma faced MA Starc and scored 0 runs. Non-striker was V Kohli.'}, {'inputs': '### Instruction: What happened in Over 0 Ball 3? \n\n### Response: In Over 0 Ball 3, RG Sharma faced MA Starc and scored 4 runs. Non-striker was V Kohli.'}, {'inputs': '### Instruction: What happened i

In [1]:
import torch
import bitsandbytes as bnb

print("Bitsandbytes version:", bnb.__version__)
print("CUDA is available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("CUDA device name:", torch.cuda.get_device_name(0))

try:
    bnb.test.test_torch_cuda()
    print("bitsandbytes test passed.")
except Exception as e:
    print("bitsandbytes test failed:", str(e))




Bitsandbytes version: 0.43.1
CUDA is available: True
CUDA device count: 1
CUDA device name: NVIDIA GeForce RTX 3050 Laptop GPU
bitsandbytes test failed: module 'bitsandbytes' has no attribute 'test'


In [1]:
import torch
from diffusers import (
    EulerDiscreteScheduler,
    MotionAdapter,
    PIAPipeline,
)
from diffusers.utils import export_to_gif, load_image
from PIL import Image

# Load the adapter and pipeline
adapter = MotionAdapter.from_pretrained("openmmlab/PIA-condition-adapter")
pipe = PIAPipeline.from_pretrained("SG161222/Realistic_Vision_V6.0_B1_noVAE", motion_adapter=adapter, torch_dtype=torch.float32)
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
pipe = pipe.to(device)

# Load and resize the image
image_path = "/kaggle/input/box-pushing/a7339c2e-e1ce-459c-8b06-1a27810d74ae.jpeg"
image = load_image(image_path)
image = image.resize((512, 512))

# Define the prompt and negative prompt
prompt = "boy should be pushing the box,and the box needs to move"
negative_prompt = "wrong white balance, dark, sketches, worst quality, low quality, deformed, distorted, disfigured, bad eyes, wrong lips, weird mouth, bad teeth, mutated hands and fingers, bad anatomy, wrong anatomy, amputation, extra limb, missing limb, floating limbs, disconnected limbs, mutation, ugly, disgusting, bad pictures, negative hand-neg"

# Create the generator based on the device
generator = torch.Generator(device).manual_seed(0)

# Generate the video
output = pipe(image=image, prompt=prompt, negative_prompt=negative_prompt, generator=generator)

# Extract frames and export to GIF
frames = output.frames[0]
export_to_gif(frames, "pia-animation.gif")


config.json:   0%|          | 0.00/374 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.67G [00:00<?, ?B/s]

KeyboardInterrupt: 